### Generating VPT image-extraction commands

This notebook generates Vizgen Post Processing Tool (VPT) commands to extract DAPI image patches from MERSCOPE imaging regions. For each region, the script reads the bounding box from `manifest.json`, calculates its centre coordinates and dimensions in microns, and constructs the corresponding `extract-image-patch` command.

The commands are printed for inspection and are not executed by the notebook. They must be run separately in a command-line environment where VPT is installed and accessible. The input and output paths should be configured for the target computing environment.

import os
import json

# Base directory containing the MERFISH dataset, organised by slide and region.
base_dir = "/group/tran3/3dImaging/zebrafishBatch2_2025Q1"

# Output directory and parameters
output_base = "/group/tran3/swei/MERFISH/MERSCOPE/data/extracted-image"
os.makedirs(output_base, exist_ok=True)
z_index = 0
stain_name = "DAPI"   # Change to "DAPI" or others if needed

In [2]:
def compute_bbox_params(bbox):
    """Calculate the centre coordinates and dimensions of a bounding box.

    Args:
        bbox: Bounding box in microns, formatted as
              [x_min, y_min, x_max, y_max].

    Returns:
        Centre x/y coordinates and width/height, all in microns.
    """
    x_min, y_min, x_max, y_max = bbox
    center_x = (x_min + x_max) / 2
    center_y = (y_min + y_max) / 2
    size_x = x_max - x_min
    size_y = y_max - y_min
    return center_x, center_y, size_x, size_y

In [3]:
# Loop through all slides and regions
for slide_name in sorted(os.listdir(base_dir)):
    slide_path = os.path.join(base_dir, slide_name)
    if not os.path.isdir(slide_path):
        continue  # skip non-folder items

    for region_name in sorted(os.listdir(slide_path)):
        region_path = os.path.join(slide_path, region_name)
        images_path = os.path.join(region_path, "images")
        manifest_path = os.path.join(images_path, "manifest.json")

        if not os.path.exists(manifest_path):
            continue
            
         # Skip regions without the required image manifest.
        try:
            # Read the region's metadata to obtain its bounding box in microns.
            with open(manifest_path, "r") as f:
                manifest = json.load(f)
            bbox = manifest.get("bbox_microns")
            if bbox is None:
                print(f"⚠️  No bbox_microns in {manifest_path}")
                continue

            # Construct the VPT command to extract the specified image patch.
            # The command is printed for review; it is not executed by this script.
            center_x, center_y, size_x, size_y = compute_bbox_params(bbox)

            # Print results
            print(f"\n {slide_name}/{region_name}")
            print(f"  bbox_microns: {bbox}")
            print(f"  center-x = {center_x}")
            print(f"  center-y = {center_y}")
            print(f"  size-x   = {size_x}")
            print(f"  size-y   = {size_y}")

            # Create vpt command
            vpt_cmd = f"""
vpt --verbose \\
--log-level 1 extract-image-patch \\
--input-images {images_path} \\
--input-micron-to-mosaic {images_path}/micron_to_mosaic_pixel_transform.csv \\
--output-patch {output_base}/{slide_name}_{region_name}_z{z_index}_DAPI.png \\
--center-x {center_x} \\
--center-y {center_y} \\
--size-x {size_x} \\
--size-y {size_y} \\
--input-z-index {z_index} \\
--green-stain-name {stain_name}
"""
            print("\n Command:")
            print(vpt_cmd)

        except Exception as e:
            print(f" Error reading {manifest_path}: {e}")



 slide1/region_0
  bbox_microns: [523.1798890755161, 6287.860023880164, 15093.025064395746, 11225.848622488831]
  center-x = 7808.102476735631
  center-y = 8756.854323184498
  size-x   = 14569.84517532023
  size-y   = 4937.988598608667

 Command:

vpt --verbose \
--log-level 1 extract-image-patch \
--input-images /group/tran3/3dImaging/zebrafishBatch2_2025Q1/slide1/region_0/images \
--input-micron-to-mosaic /group/tran3/3dImaging/zebrafishBatch2_2025Q1/slide1/region_0/images/micron_to_mosaic_pixel_transform.csv \
--output-patch /group/tran3/swei/MERFISH/MERSCOPE/data/extracted-image/slide1_region_0_z0_DAPI.png \
--center-x 7808.102476735631 \
--center-y 8756.854323184498 \
--size-x 14569.84517532023 \
--size-y 4937.988598608667 \
--input-z-index 0 \
--green-stain-name DAPI


 slide1/region_1
  bbox_microns: [-31.98764456714519, -151.48217376911967, 14530.152905191517, 5021.568]
  center-x = 7249.082630312186
  center-y = 2435.04291311544
  size-x   = 14562.140549758662
  size-y   = 51